# Module 3.4 — Run the QLoRA Fine-Tune
**DeskMate SLM Curriculum · Phase 3 · Notebook 18**

**Two-phase approach:**
1. **Smoke test** (free Colab T4, ~5 min) — 200 examples, 50 steps, confirm loss drops
2. **Full run** (rented A100, ~1–2 hrs) — 3 epochs on the full SFT dataset

Always run the smoke test first. Never pay for GPU time before verifying the setup works.

Read `3.4_qlora_finetune.md` for full theory.

---

## Step 0 — Install

In [ ]:
%%capture
!pip install -q bitsandbytes==0.43.3 peft==0.12.0 trl==0.10.1 \
               transformers==4.44.0 datasets==2.21.0 torch==2.3.1 accelerate==0.34.0

In [ ]:
import random, json, pathlib, time
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_from_disk, Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, TaskType, PeftModel, get_peft_model
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_GPU = device == 'cuda'
print(f'Device : {device}')
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}  ({props.total_memory/1e9:.1f} GB VRAM)')
else:
    print('WARNING: No GPU detected. Smoke test will run in CPU-compatible fallback mode.')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Runtime : {RUNTIME}')

## Step 2 — Load SFT Dataset

In [ ]:
import pandas as pd

sft_path = DATA_PROC / 'sft_dataset'
if sft_path.exists():
    sft_ds = load_from_disk(str(sft_path))
    print(f'Loaded SFT dataset:')
    for split, ds in sft_ds.items():
        print(f'  {split}: {len(ds)} examples')
else:
    print('SFT dataset not found — building placeholder (run Module 3.2 first)')
    INTENTS = ['account_access','billing_dispute','technical_bug',
               'usage_question','outage_report']
    rows = []
    for i in range(200):
        intent = INTENTS[i % len(INTENTS)]
        rows.append({
            'ticket': f'I have a {intent.replace("_"," ")} issue #{i}.',
            'reply':  f'Thank you for contacting DeskMate. We will resolve your '
                      f'{intent.replace("_"," ")} issue within 24 hours.',
            'intent': intent, 'context': None, 'source': 'placeholder',
        })
    df = pd.DataFrame(rows)
    sft_ds = DatasetDict({
        'train': Dataset.from_pandas(df.iloc[:160]),
        'val':   Dataset.from_pandas(df.iloc[160:]),
    })

# Inspect one example
ex = sft_ds['train'][0]
print('\nFirst training example keys:', list(ex.keys()))
if 'input_ids' in ex:
    print(f'input_ids length: {len(ex["input_ids"])}')
    print(f'labels length   : {len(ex["labels"])}')
    masked = sum(1 for l in ex['labels'] if l == -100)
    print(f'masked (-100)   : {masked} ({masked/len(ex["labels"])*100:.0f}%)')

## Step 3 — Load Tokenizer & Model Config

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
print(f'Base model: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chat template (same as Module 3.2)
if not tokenizer.chat_template:
    tokenizer.chat_template = (
        '{%- for message in messages %}'
        '<|im_start|>{{ message["role"] }}\n{{ message["content"] }}<|im_end|>\n'
        '{%- endfor %}'
        '{%- if add_generation_prompt %}<|im_start|>assistant\n{%- endif %}'
    )

print(f'Tokenizer: vocab_size={tokenizer.vocab_size:,}')
print(f'EOS token: {tokenizer.eos_token!r}  pad: {tokenizer.pad_token!r}')

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules='all-linear',
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

print('BitsAndBytesConfig: NF4, double-quant, bfloat16 compute')
print(f'LoRA: r={lora_config.r}, alpha={lora_config.lora_alpha}, '
       f'dropout={lora_config.lora_dropout}')

## Step 4 — Load 4-bit Model & Apply LoRA

In [ ]:
if HAS_GPU:
    if HAS_GPU:
        mem_before = torch.cuda.memory_allocated(0) / 1e9
    print(f'Loading {MODEL_NAME} in 4-bit ...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model.enable_input_require_grads()
    mem_after_load = torch.cuda.memory_allocated(0) / 1e9
    print(f'Model loaded. VRAM used: {mem_after_load:.2f} GB')
    print(f'  (base load delta: {mem_after_load - mem_before:.2f} GB)')
else:
    print('No GPU — loading in fp32 for CPU smoke test (very slow, for structure check only)')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float32)
    model.config.use_cache = False
    model.enable_input_require_grads()

In [ ]:
# Apply LoRA and inspect trainable parameters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if HAS_GPU:
    mem_after_lora = torch.cuda.memory_allocated(0) / 1e9
    print(f'VRAM after LoRA: {mem_after_lora:.2f} GB')

## Step 5 — DataCollator & Dataset Prep

In [ ]:
RESPONSE_TEMPLATE = '<|im_start|>assistant\n'
collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)

# If dataset has raw text columns, tokenize here
# If it was pre-tokenized in Module 3.2, set format directly
train_ds = sft_ds['train']
val_ds   = sft_ds['val']

if 'input_ids' not in train_ds.column_names:
    print('Pre-tokenizing on-the-fly (dataset lacks input_ids) ...')
    SYSTEM_MSG = (
        'You are DeskMate, a concise customer support assistant. '
        'Respond in 2-4 sentences.'
    )
    def tokenize_fn(ex):
        ticket   = ex.get('ticket', ex.get('text', ''))
        reply    = ex.get('reply', '')
        context  = ex.get('context')
        user_msg = (('Context: ' + context + '\n\n') if context else '') + 'Ticket: ' + ticket
        msgs = [{'role':'system','content':SYSTEM_MSG},
                {'role':'user',  'content':user_msg},
                {'role':'assistant','content':reply}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=False)
        enc  = tokenizer(text, truncation=True, max_length=512, padding=False)
        enc['labels'] = list(enc['input_ids'])
        return enc
    drop = [c for c in train_ds.column_names
            if c not in ('input_ids','attention_mask','labels')]
    train_ds = train_ds.map(tokenize_fn, batched=False, remove_columns=drop)
    val_ds   = val_ds.map(tokenize_fn,   batched=False, remove_columns=drop)

train_ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
val_ds.set_format(  'torch', columns=['input_ids','attention_mask','labels'])
print(f'Ready: train={len(train_ds)}  val={len(val_ds)}')

## Step 6 — Smoke Test (50 Steps)

**Decision gate:** if loss is not clearly decreasing by step 50, stop.
Do not proceed to the full run until this passes.

In [ ]:
SMOKE_N = min(200, len(train_ds))
smoke_ds = train_ds.select(range(SMOKE_N))

smoke_args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / 'deskmate-qlora-smoke'),
    num_train_epochs            = 1,
    max_steps                   = 50,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    gradient_checkpointing      = True,
    learning_rate               = 2e-4,
    warmup_steps                = 5,
    bf16                        = HAS_GPU,
    fp16                        = False,
    logging_steps               = 10,
    save_steps                  = 50,
    report_to                   = 'none',
    seed                        = SEED,
)

smoke_trainer = SFTTrainer(
    model          = model,
    args           = smoke_args,
    train_dataset  = smoke_ds,
    data_collator  = collator,
    tokenizer      = tokenizer,
    max_seq_length = 512,
)

print('Starting smoke test (50 steps) ...')
smoke_trainer.train()
print('Smoke test complete.')

In [ ]:
# Plot smoke test loss and make go/no-go decision
logs = [x for x in smoke_trainer.state.log_history if 'loss' in x]

if logs:
    steps  = [x['step'] for x in logs]
    losses = [x['loss'] for x in logs]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(steps, losses, 'o-', color='#4C72B0')
    ax.set_xlabel('Step'); ax.set_ylabel('Train Loss')
    ax.set_title('Smoke Test — Training Loss (should trend down)')
    plt.tight_layout(); plt.show()

    first_loss = losses[0]
    last_loss  = losses[-1]
    drop_pct   = (first_loss - last_loss) / first_loss * 100
    print(f'Loss: {first_loss:.4f} → {last_loss:.4f}  ({drop_pct:+.1f}%)')
    if drop_pct > 5:
        print('PASS — loss is dropping. Safe to proceed to full run.')
    elif drop_pct > 0:
        print('MARGINAL — slight drop. Check learning rate before full run.')
    else:
        print('FAIL — loss not dropping. Debug before spending GPU budget.')
        print('Check: chat template, loss mask, learning rate, data format.')
else:
    print('No loss logs found — check logging_steps setting.')

## Step 7 — Before Samples (Base Model)

In [ ]:
SAMPLE_TICKETS = [
    'I cannot log in to my account. My password reset email never arrived.',
    'I was charged twice for the Pro plan last month. Please help.',
    'The CSV export button throws a 500 error on every browser I try.',
]
SYSTEM_MSG = 'You are DeskMate, a concise support assistant. Respond in 2-4 sentences.'

def generate_reply(m, tok, ticket, max_new_tokens=120):
    msgs = [{'role':'system','content':SYSTEM_MSG},
            {'role':'user',  'content':'Ticket: ' + ticket}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(m.device)
    m.eval()
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tok.eos_token_id)
    new_toks = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()

print('=== BEFORE fine-tuning (smoke-test checkpoint) ===')
for ticket in SAMPLE_TICKETS:
    reply = generate_reply(smoke_trainer.model, tokenizer, ticket)
    print(f'Ticket: {ticket}')
    print(f'Reply : {reply}')
    print()

## Step 8 — Full Run

Switch `RUN_FULL = True` when:
- Smoke test passed (loss dropped)
- You are on a rented A100 or have sufficient T4 time

On free Colab T4, the full run takes ~60–90 min for a few thousand examples.

In [ ]:
RUN_FULL = False   # Set True to launch full training

if RUN_FULL:
    full_args = TrainingArguments(
        output_dir                  = str(MODELS_DIR / 'deskmate-qlora-full'),
        num_train_epochs            = 3,
        per_device_train_batch_size = 4 if HAS_GPU else 1,
        gradient_accumulation_steps = 4,
        gradient_checkpointing      = True,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = 'cosine',
        bf16                        = HAS_GPU,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        logging_steps               = 25,
        report_to                   = 'none',
        seed                        = SEED,
    )
    full_trainer = SFTTrainer(
        model          = model,
        args           = full_args,
        train_dataset  = train_ds,
        eval_dataset   = val_ds,
        data_collator  = collator,
        tokenizer      = tokenizer,
        max_seq_length = 512,
    )
    print('Starting full run ...')
    full_trainer.train()
    trained_model = full_trainer.model
    print('Full run complete.')
else:
    trained_model = smoke_trainer.model
    print('Skipped full run (RUN_FULL=False). Using smoke-test checkpoint.')
    print('Set RUN_FULL=True and re-run on A100 for production quality.')

## Step 9 — Save LoRA Adapter

In [ ]:
adapter_path = MODELS_DIR / 'deskmate-qlora-adapter'
trained_model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

# Report adapter size
import os
total_bytes = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(adapter_path)
    for f in files
)
print(f'Adapter saved: {adapter_path}')
print(f'Adapter size : {total_bytes/1e6:.1f} MB  '
       f'(vs ~3 GB for full bf16 model)')

## Step 10 — After Samples

In [ ]:
print('=== AFTER fine-tuning ===')
for ticket in SAMPLE_TICKETS:
    reply = generate_reply(trained_model, tokenizer, ticket)
    print(f'Ticket: {ticket}')
    print(f'Reply : {reply}')
    print()

## Step 11 — Merge Adapter (Optional)

Merging folds LoRA weights into the base model — faster inference, no LoRA overhead.
Only possible on GPU (requires dequantising the NF4 base to bf16 first).

In [ ]:
MERGE = HAS_GPU   # Can only merge on GPU

if MERGE:
    print('Merging LoRA adapter into base weights ...')
    # Reload base in bf16 (not 4-bit) for merging
    base_bf16 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
    )
    peft_merged = PeftModel.from_pretrained(base_bf16, str(adapter_path))
    merged = peft_merged.merge_and_unload()
    merged_path = MODELS_DIR / 'deskmate-merged'
    merged.save_pretrained(str(merged_path))
    tokenizer.save_pretrained(str(merged_path))
    print(f'Merged model saved: {merged_path}')
    merged_bytes = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(merged_path) for f in files)
    print(f'Merged size: {merged_bytes/1e9:.2f} GB')
else:
    print('Merge skipped (GPU required). Load adapter at inference time instead.')
    print('At inference: PeftModel.from_pretrained(base, adapter_path)')

## Step 12 — VRAM Usage Report

In [ ]:
if HAS_GPU:
    peak = torch.cuda.max_memory_allocated(0) / 1e9
    curr = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('VRAM Usage Summary')
    print(f'  Total VRAM     : {total:.1f} GB')
    print(f'  Peak (training): {peak:.2f} GB ({peak/total*100:.0f}% of total)')
    print(f'  Current        : {curr:.2f} GB')
    print(f'  Headroom       : {total - peak:.2f} GB')
    if peak < total * 0.85:
        print('  Status: OK — could increase batch size for faster training')
    elif peak < total:
        print('  Status: TIGHT — reduce batch_size or max_seq_length if OOM errors appear')
    else:
        print('  Status: OOM — reduce batch_size, max_seq_length, or increase grad_accum')
else:
    print('VRAM report: N/A (CPU run)')
    print('Expected peak on T4 with these settings: ~3.5-5 GB')

## Checkpoint

> **Which two memory-saving techniques let this fit in 16 GB, and what do they cost you?**

Write your answer below.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| LoRA adapter | `models/deskmate-qlora-adapter/` |
| Merged model (if GPU) | `models/deskmate-merged/` |

**What you've built:** a fine-tuned DeskMate decoder adapter trained with QLoRA.

**Next:** Module 3.5 — Is bigger actually better?
Rent an A100, run a 7B QLoRA, and compare quality / latency / cost against the 1.5B.